# 02_train_baselines

Huấn luyện 3 mô hình LeNet-5 FP32 (seed 0, 1, 2) trên Fashion-MNIST làm mô hình gốc
cho các thí nghiệm lượng tử hóa sau huấn luyện (Thí nghiệm B, C).

Cấu hình: SGD momentum 0.9, lr 0.05 giảm theo cosine, batch 128, 15 epoch,
không weight decay, không tăng cường dữ liệu.

## Mục tiêu và cách đọc

Huấn luyện ba mô hình FP32 làm mốc. Dữ liệu nằm trong data/. Chạy lại sẽ huấn luyện lại và thay mô hình chuẩn; có bản sao trước chạy trong before_run/.

## Cách sử dụng

1. Chọn kernel Python có `ipykernel`, `numpy`, `torch`, `matplotlib`.
2. Giữ `RUN_EXPERIMENT = False` và chọn **Run All** để xem dữ liệu sẵn có.
3. Đổi thành `True` nếu muốn thực hiện lại; chạy tuần tự các ô từ đầu.
4. **Ctrl+S / Cmd+S** lưu toàn bộ output vào notebook. Báo cáo HTML và CSV được xuất tự động ở ô cuối.

**Lưu ý:** các hình B/C/D là file đã lưu; sau khi đổi kết quả, chạy notebook 08 để vẽ lại trước khi dùng hình trong bài. Thời điểm xuất báo cáo không phải thời điểm thực hiện thí nghiệm gốc.


## 1. Thiết lập và chọn chế độ

Cài môi trường một lần theo `HUONG_DAN_NOTEBOOK.md`. Mỗi lần chạy tạo thư mục riêng trong `results/notebook_runs/`. Khi chạy lại thí nghiệm, các kết quả chuẩn và checkpoint hiện có được sao lưu trước.

In [ ]:
from pathlib import Path
import os
import sys

# Tìm 02_demo dù kernel bắt đầu ở thư mục workspace hoặc thư mục notebook.
candidates = [Path.cwd(), *Path.cwd().parents]
DEMO_ROOT = next((base / suffix for base in candidates
                  for suffix in ("", "02_demo", "Gk_XLDL/02_demo")
                  if (base / suffix / "posit_lab/numformats.py").is_file()), None)
if DEMO_ROOT is None:
    raise FileNotFoundError("Mở notebook trong workspace chứa Gk_XLDL/02_demo.")
DEMO_ROOT = DEMO_ROOT.resolve()
os.chdir(DEMO_ROOT)
if str(DEMO_ROOT) not in sys.path:
    sys.path.insert(0, str(DEMO_ROOT))
from notebook_tools.reporting import NotebookRun

# False: xem số liệu sẵn có. True: chạy code thí nghiệm bên dưới.
RUN_EXPERIMENT = False
SCRIPT_ARGS = []
__file__ = str(DEMO_ROOT / '02_train_baselines.py')
__package__ = None
session = NotebookRun(DEMO_ROOT, '02_train_baselines.py', RUN_EXPERIMENT, SCRIPT_ARGS)
print("Thư mục lưu lần này:", session.out)

# Cell magic giữ code thẳng hàng; vẫn áp dụng chế độ chạy và lưu nhật ký.
from IPython.core.magic import register_cell_magic
import __future__

@register_cell_magic
def experiment(line, cell):
    if not RUN_EXPERIMENT:
        print("Đã bỏ qua tính toán; đặt RUN_EXPERIMENT = True để chạy ô này.")
        return
    with session.execute():
        code = compile(cell, str(session.source), "exec",
                       flags=__future__.annotations.compiler_flag)
        exec(code, get_ipython().user_ns)


## 2. Chuẩn bị / tính toán / lưu dữ liệu theo thứ tự script

Mã chuyển nguyên từ `02_train_baselines.py`. Ô chỉ thực thi khi `RUN_EXPERIMENT = True`; giữ thứ tự để các biến và hàm được khởi tạo đúng.

In [ ]:
%%experiment
"""
Huấn luyện 3 mô hình LeNet-5 FP32 (seed 0, 1, 2) trên Fashion-MNIST làm mô hình gốc
cho các thí nghiệm lượng tử hóa sau huấn luyện (Thí nghiệm B, C).

Cấu hình: SGD momentum 0.9, lr 0.05 giảm theo cosine, batch 128, 15 epoch,
không weight decay, không tăng cường dữ liệu.
"""
import json
import math
import os
import time

import torch
import torch.nn.functional as F

from posit_lab.data import load_fashion_mnist
from posit_lab.model import LeNet5, evaluate

HERE = os.path.dirname(os.path.abspath(__file__))
CKPT = os.path.join(HERE, "results", "checkpoints")
os.makedirs(CKPT, exist_ok=True)
torch.set_num_threads(4)

EPOCHS, BS, LR = 15, 128, 0.05
xtr, ytr = load_fashion_mnist("train")
xte, yte = load_fashion_mnist("test")
summary = {}
for seed in [0, 1, 2]:
    torch.manual_seed(seed)
    model = LeNet5()
    opt = torch.optim.SGD(model.parameters(), lr=LR, momentum=0.9)
    steps = EPOCHS * math.ceil(len(xtr) / BS)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    t0 = time.time()
    for ep in range(EPOCHS):
        model.train()
        perm = torch.randperm(len(xtr))
        for i in range(0, len(xtr), BS):
            idx = perm[i:i + BS]
            loss = F.cross_entropy(model(xtr[idx]), ytr[idx])
            opt.zero_grad()
            loss.backward()
            opt.step()
            sched.step()
        acc = evaluate(model, xte, yte)
        print(f"seed {seed} epoch {ep + 1:2d} loss {loss.item():.4f} test acc {acc:.2f}% "
              f"({time.time() - t0:.0f}s)", flush=True)
    torch.save(model.state_dict(), os.path.join(CKPT, f"lenet5_fp32_seed{seed}.pt"))
    summary[f"seed{seed}"] = {"test_acc": acc, "train_time_s": time.time() - t0}



## 3. Chuẩn bị / tính toán / lưu dữ liệu theo thứ tự script

Mã chuyển nguyên từ `02_train_baselines.py`. Ô chỉ thực thi khi `RUN_EXPERIMENT = True`; giữ thứ tự để các biến và hàm được khởi tạo đúng.

In [ ]:
%%experiment
with open(os.path.join(HERE, "results", "01_baselines.json"), "w") as f:
    json.dump(summary, f, indent=2)
print(summary)


## Kết quả, bảng số liệu và hình

Ô cuối đọc đầy đủ các trường JSON, xuất CSV UTF-8 cho Excel, lưu JSON gốc, nhúng hình vào HTML để mở độc lập. Các bảng dài có thanh cuộn. Khi viết bài, ghi rõ cấu hình, seed, tập dữ liệu và giới hạn mô phỏng.

In [ ]:
session.report(['01_baselines.json'], [])

metric,value
test_acc,90.76
train_time_s,173.19456911087036
metric,value
test_acc,90.69
train_time_s,166.76907920837402
metric,value
test_acc,91.37
train_time_s,153.2909140586853
